In [1]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

DATA = Path("../data")
OUTPUTS = Path("../outputs")

In [2]:
train = pl.read_parquet(OUTPUTS / "train_model_ready.parquet")
new_cont = pl.read_parquet(OUTPUTS / "features_contamination_v2.parquet")
labels = pl.read_parquet(DATA / "train_labels.parquet")

train = (
    train
    .join(new_cont.select([
        "account_id",
        "new_mule_network_cp_count",
        "new_mule_cp_weighted_score",
        "new_contamination_rate"
    ]), on="account_id", how="left")
    .with_columns([
        pl.col("new_mule_network_cp_count").fill_null(0),
        pl.col("new_mule_cp_weighted_score").fill_null(0),
        pl.col("new_contamination_rate").fill_null(0),
    ])
)

feature_cols = [c for c in train.columns if c not in ["account_id", "is_mule"]]

X = train.select(feature_cols).to_pandas()
y = train["is_mule"].to_pandas()

print("Shape:", X.shape)
print("Mule rate:", y.mean())

Shape: (96091, 73)
Mule rate: 0.027921449459366643


In [3]:
# Load test with same feature construction as train

test = pl.read_parquet(DATA / "test_accounts.parquet")
new_cont = pl.read_parquet(OUTPUTS / "features_contamination_v2.parquet")
master = pl.read_parquet(OUTPUTS / "master_features_all.parquet")
accounts = pl.read_parquet(DATA / "accounts.parquet")
branch_feat = pl.read_parquet(OUTPUTS / "features_branch.parquet")
mcc = pl.read_parquet(OUTPUTS / "features_mcc.parquet")

test_full = (
    test
    .join(master.drop(["branch_code"]), on="account_id", how="left")
    .join(accounts.select(["account_id", "branch_code"]), on="account_id", how="left")
    .join(branch_feat.select(["branch_code", "branch_turnover", "branch_asset_size"]), on="branch_code", how="left")
    .join(mcc.select(["account_id", "max_mcc_zscore", "avg_mcc_zscore", "max_abs_mcc_zscore"]), on="account_id", how="left")
    .join(new_cont.select([
        "account_id",
        "new_mule_network_cp_count",
        "new_mule_cp_weighted_score",
        "new_contamination_rate"
    ]), on="account_id", how="left")
)

# Now define numeric columns OUTSIDE the parentheses
numeric_cols = [
    c for c, dt in zip(test_full.columns, test_full.dtypes)
    if dt in [pl.Int8, pl.Int16, pl.Int32, pl.Int64,
              pl.Float32, pl.Float64]
]

test_full = test_full.with_columns(
    pl.col(numeric_cols).fill_null(0)
)

X_test = test_full.select(feature_cols).to_pandas()

print("Train shape:", X.shape)
print("Test shape:", X_test.shape)

Train shape: (96091, 73)
Test shape: (64062, 73)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

print("\n==============================")
print("ADVERSARIAL VALIDATION")
print("==============================")

# Combine train + test
X_adv = pd.concat([X, X_test], ignore_index=True)

y_adv = np.concatenate([
    np.zeros(len(X)),   # train = 0
    np.ones(len(X_test))  # test = 1
])

# Train/val split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_adv, y_adv,
    test_size=0.2,
    stratify=y_adv,
    random_state=42
)

model_adv = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_adv.fit(X_tr, y_tr)

val_preds = model_adv.predict_proba(X_val)[:, 1]
auc_adv = roc_auc_score(y_val, val_preds)

print(f"Adversarial AUC: {auc_adv:.4f}")


ADVERSARIAL VALIDATION
[LightGBM] [Info] Number of positive: 51249, number of negative: 76873
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006669 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14358
[LightGBM] [Info] Number of data points in the train set: 128122, number of used features: 73
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400002 -> initscore=-0.405459
[LightGBM] [Info] Start training from score -0.405459
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

In [5]:
print("\n==============================")
print("TOP DRIFT FEATURES")
print("==============================")

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": model_adv.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df.head(20).to_string(index=False))


TOP DRIFT FEATURES
                   feature  importance
                  upi_rate        1219
days_since_passbook_update        1010
            interbank_rate         681
  days_since_mobile_update         642
                  atm_rate         617
 new_mule_network_cp_count         558
        contamination_rate         485
new_mule_cp_weighted_score         436
   volume_to_balance_ratio         356
    new_contamination_rate         311
                 atm_count         311
 days_since_address_update         277
         customer_age_days         250
          account_age_days         248
  relationship_tenure_days         248
           late_night_rate         223
         branch_asset_size         220
              txn_velocity         217
          active_span_days         215
               burst_ratio         212


In [6]:
drift_features = [
    "upi_rate",
    "interbank_rate",
    "atm_rate",
    "atm_count",
    "days_since_passbook_update",
    "days_since_mobile_update",
    "days_since_address_update",
    "contamination_rate"
]

stable_features = [f for f in feature_cols if f not in drift_features]

print("Original feature count:", len(feature_cols))
print("Stable feature count:", len(stable_features))

Original feature count: 73
Stable feature count: 65


In [7]:
print("\n==============================")
print("ADVERSARIAL VALIDATION (STABLE FEATURES)")
print("==============================")

X_adv_stable = pd.concat([
    X[stable_features],
    X_test[stable_features]
], ignore_index=True)

y_adv = np.concatenate([
    np.zeros(len(X)),
    np.ones(len(X_test))
])

X_tr, X_val, y_tr, y_val = train_test_split(
    X_adv_stable, y_adv,
    test_size=0.2,
    stratify=y_adv,
    random_state=42
)

model_adv_stable = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_adv_stable.fit(X_tr, y_tr)

val_preds = model_adv_stable.predict_proba(X_val)[:, 1]
auc_adv_stable = roc_auc_score(y_val, val_preds)

print(f"Adversarial AUC (stable features): {auc_adv_stable:.4f}")


ADVERSARIAL VALIDATION (STABLE FEATURES)
[LightGBM] [Info] Number of positive: 51249, number of negative: 76873
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013416 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12539
[LightGBM] [Info] Number of data points in the train set: 128122, number of used features: 65
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400002 -> initscore=-0.405459
[LightGBM] [Info] Start training from score -0.405459
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [

In [8]:
print("\n==============================")
print("TOP DRIFT FEATURES (STABLE SET)")
print("==============================")

importance_df_stable = pd.DataFrame({
    "feature": stable_features,
    "importance": model_adv_stable.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df_stable.head(20).to_string(index=False))


TOP DRIFT FEATURES (STABLE SET)
                   feature  importance
          upi_credit_count        1987
                imps_count        1205
           total_txn_count         834
                neft_count         700
       standing_instr_rate         420
    mule_cp_weighted_score         365
    new_contamination_rate         346
   volume_to_balance_ratio         324
               debit_count         307
           upi_debit_count         281
              avg_txn_hour         273
           late_night_rate         264
new_mule_cp_weighted_score         260
 new_mule_network_cp_count         250
         customer_age_days         229
  relationship_tenure_days         224
              credit_count         223
           unique_channels         220
           branch_turnover         219
      counterparty_entropy         214


In [9]:
volume_features = [
    "total_txn_count",
    "upi_credit_count",
    "upi_debit_count",
    "imps_count",
    "neft_count",
    "debit_count",
    "credit_count"
]

for col in volume_features:
    if col in X.columns:
        X[col] = np.log1p(X[col])
        X_test[col] = np.log1p(X_test[col])

print("Log transformation applied to volume features.")

Log transformation applied to volume features.


In [10]:
print("\n==============================")
print("ADVERSARIAL VALIDATION (STABLE FEATURES)")
print("==============================")

X_adv_stable = pd.concat([
    X[stable_features],
    X_test[stable_features]
], ignore_index=True)

y_adv = np.concatenate([
    np.zeros(len(X)),
    np.ones(len(X_test))
])

X_tr, X_val, y_tr, y_val = train_test_split(
    X_adv_stable, y_adv,
    test_size=0.2,
    stratify=y_adv,
    random_state=42
)

model_adv_stable = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_adv_stable.fit(X_tr, y_tr)

val_preds = model_adv_stable.predict_proba(X_val)[:, 1]
auc_adv_stable = roc_auc_score(y_val, val_preds)

print(f"Adversarial AUC (stable features): {auc_adv_stable:.4f}")


ADVERSARIAL VALIDATION (STABLE FEATURES)
[LightGBM] [Info] Number of positive: 51249, number of negative: 76873
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017538 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12539
[LightGBM] [Info] Number of data points in the train set: 128122, number of used features: 65
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400002 -> initscore=-0.405459
[LightGBM] [Info] Start training from score -0.405459
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [

In [11]:
more_drift = [
    "upi_credit_count",
    "imps_count",
    "total_txn_count",
    "neft_count"
]

stable_features_v2 = [
    f for f in stable_features if f not in more_drift
]

print("New feature count:", len(stable_features_v2))

New feature count: 61


In [12]:
print("\n==============================")
print("ADVERSARIAL VALIDATION (STABLE FEATURES)")
print("==============================")

X_adv_stable = pd.concat([
    X[stable_features],
    X_test[stable_features]
], ignore_index=True)

y_adv = np.concatenate([
    np.zeros(len(X)),
    np.ones(len(X_test))
])

X_tr, X_val, y_tr, y_val = train_test_split(
    X_adv_stable, y_adv,
    test_size=0.2,
    stratify=y_adv,
    random_state=42
)

model_adv_stable = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_adv_stable.fit(X_tr, y_tr)

val_preds = model_adv_stable.predict_proba(X_val)[:, 1]
auc_adv_stable = roc_auc_score(y_val, val_preds)

print(f"Adversarial AUC (stable features): {auc_adv_stable:.4f}")


ADVERSARIAL VALIDATION (STABLE FEATURES)
[LightGBM] [Info] Number of positive: 51249, number of negative: 76873
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007505 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12539
[LightGBM] [Info] Number of data points in the train set: 128122, number of used features: 65
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400002 -> initscore=-0.405459
[LightGBM] [Info] Start training from score -0.405459
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No

In [13]:
print("\n==============================")
print("RETRAINING WITH STABLE FEATURES")
print("==============================")

# Prepare data
X_stable = X[stable_features].copy()
X_test_stable = X_test[stable_features].copy()

print("Stable train shape:", X_stable.shape)
print("Stable test shape:", X_test_stable.shape)


RETRAINING WITH STABLE FEATURES
Stable train shape: (96091, 65)
Stable test shape: (64062, 65)


In [14]:
# Recompute scale_pos_weight
pos = y.sum()
neg = len(y) - pos
scale_pos_weight = neg / pos

In [15]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("CV ready.")

CV ready.


In [16]:
print("\n==============================")
print("TRAINING STABLE MODEL")
print("==============================")

lgb_stable = lgb.LGBMClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.7,
    scale_pos_weight=scale_pos_weight,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

oof_stable = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(cv.split(X_stable, y)):
    X_tr, X_val = X_stable.iloc[train_idx], X_stable.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgb_stable.fit(X_tr, y_tr)
    oof_stable[val_idx] = lgb_stable.predict_proba(X_val)[:, 1]

    fold_auc = roc_auc_score(y_val, oof_stable[val_idx])
    print(f"Fold {fold+1}: AUC={fold_auc:.4f}")

stable_auc = roc_auc_score(y, oof_stable)
print(f"\nStable OOF AUC: {stable_auc:.4f}")


TRAINING STABLE MODEL
Fold 1: AUC=0.9987
Fold 2: AUC=0.9981
Fold 3: AUC=0.9978


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import f1_score

print("\n==============================")
print("FINDING BEST THRESHOLD (STABLE)")
print("==============================")

thresholds = np.linspace(0.1, 0.9, 200)
best_thresh = 0
best_f1 = 0

for t in thresholds:
    preds = (oof_stable >= t).astype(int)
    f1 = f1_score(y, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"Best Threshold: {best_thresh:.3f}")
print(f"Best F1 Score: {best_f1:.4f}")


FINDING BEST THRESHOLD (STABLE)
Best Threshold: 0.836
Best F1 Score: 0.9630


In [ ]:
print("\n==============================")
print("TEST DISTRIBUTION (STABLE)")
print("==============================")

lgb_stable.fit(X_stable, y)
test_probs_stable = lgb_stable.predict_proba(X_test_stable)[:, 1]

for t in [0.3, best_thresh]:
    count = (test_probs_stable >= t).sum()
    print(f"> {t:.3f}: {count} ({count/len(test_probs_stable):.2%})")

print(f"\nExpected mules (2.79% prior): {int(64062 * 0.0279)}")


TEST DISTRIBUTION (STABLE)
> 0.300: 614 (0.96%)
> 0.836: 503 (0.79%)

Expected mules (2.79% prior): 1787


In [ ]:
no_network = [
    "new_mule_network_cp_count",
    "new_mule_cp_weighted_score",
    "new_contamination_rate",
    "mule_cp_weighted_score"
]

stable_no_network = [
    f for f in stable_features if f not in no_network
]

print("Feature count without network:", len(stable_no_network))

Feature count without network: 61


In [ ]:
print("\n==============================")
print("TRAINING STABLE MODEL (NO NETWORK)")
print("==============================")

X_nn = X[stable_no_network].copy()
X_test_nn = X_test[stable_no_network].copy()

lgb_nn = lgb.LGBMClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.7,
    scale_pos_weight=scale_pos_weight,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

oof_nn = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(cv.split(X_nn, y)):
    X_tr, X_val = X_nn.iloc[train_idx], X_nn.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgb_nn.fit(X_tr, y_tr)
    oof_nn[val_idx] = lgb_nn.predict_proba(X_val)[:, 1]

    fold_auc = roc_auc_score(y_val, oof_nn[val_idx])
    print(f"Fold {fold+1}: AUC={fold_auc:.4f}")

nn_auc = roc_auc_score(y, oof_nn)
print(f"\nOOF AUC (no network): {nn_auc:.4f}")


TRAINING STABLE MODEL (NO NETWORK)
Fold 1: AUC=0.9988
Fold 2: AUC=0.9977
Fold 3: AUC=0.9989
Fold 4: AUC=0.9987
Fold 5: AUC=0.9984

OOF AUC (no network): 0.9985
